# Day 13 · Bounding Box Foundations

**Objective:** Write a PyTorch Intersection over Union (IoU) function and test it using an apple ground-truth annotation and simulated predictions.

## 1. Download apple annotations

Download the Roboflow fruit-and-veggie dataset in YOLO format. We will select an image containing an apple and use its annotation as the ground-truth box.

In [ ]:
!pip -q install roboflow ultralytics

from getpass import getpass
from roboflow import Roboflow
import yaml
from pathlib import Path

ROBOFLOW_API_KEY = getpass("Enter your Roboflow API key: ")
rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace("projects-gmksb").project("fruit-and-veggie")
dataset = project.version(4).download("yolov8")
DATA_DIR = Path(dataset.location)

with open(DATA_DIR / "data.yaml", "r") as file:
    data_config = yaml.safe_load(file)

class_names = data_config["names"]
if isinstance(class_names, dict):
    class_names = [class_names[i] for i in sorted(class_names)]

print("Dataset:", DATA_DIR)
print("Classes:", class_names)

In [ ]:
import torch
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as patches

# Find the apple class ID from the dataset labels.
apple_ids = [i for i, name in enumerate(class_names) if "apple" in name.lower()]
if not apple_ids:
    raise ValueError("No apple class found in this dataset.")
apple_id = apple_ids[0]

# Find an image whose YOLO label file contains an apple.
image_extensions = {".jpg", ".jpeg", ".png"}
image_path, label_path = None, None

for label_file in DATA_DIR.rglob("*.txt"):
    if label_file.name in {"classes.txt"}:
        continue
    matching_image = None
    for ext in image_extensions:
        candidate = label_file.parent.parent / "images" / label_file.name.replace(".txt", ext)
        if candidate.exists():
            matching_image = candidate
            break
    if matching_image is None:
        continue
    rows = [line.split() for line in label_file.read_text().splitlines() if line.strip()]
    apple_rows = [row for row in rows if int(row[0]) == apple_id]
    if apple_rows:
        image_path, label_path, annotation_rows = matching_image, label_file, apple_rows
        break

if image_path is None:
    raise FileNotFoundError("Could not locate an image with an apple annotation.")

image = Image.open(image_path).convert("RGB")
width, height = image.size
class_id, xc, yc, bw, bh = map(float, annotation_rows[0])
ground_truth = torch.tensor([[
    (xc - bw / 2) * width,
    (yc - bh / 2) * height,
    (xc + bw / 2) * width,
    (yc + bh / 2) * height
]], dtype=torch.float32)

print("Image:", image_path.name)
print("Image size:", image.size)
print("Ground-truth apple box (x1, y1, x2, y2):", ground_truth.tolist()[0])

## 2. Implement IoU in PyTorch

IoU measures how much a predicted box overlaps the ground-truth box.

\[
IoU = \frac{Area\ of\ Intersection}{Area\ of\ Union}
\]

Boxes use the format `[x1, y1, x2, y2]`.

In [ ]:
def calculate_iou(boxes1, boxes2):
    # Intersection rectangle
    x1 = torch.maximum(boxes1[:, 0], boxes2[:, 0])
    y1 = torch.maximum(boxes1[:, 1], boxes2[:, 1])
    x2 = torch.minimum(boxes1[:, 2], boxes2[:, 2])
    y2 = torch.minimum(boxes1[:, 3], boxes2[:, 3])

    intersection = (x2 - x1).clamp(min=0) * (y2 - y1).clamp(min=0)

    area1 = (boxes1[:, 2] - boxes1[:, 0]).clamp(min=0) * (boxes1[:, 3] - boxes1[:, 1]).clamp(min=0)
    area2 = (boxes2[:, 2] - boxes2[:, 0]).clamp(min=0) * (boxes2[:, 3] - boxes2[:, 1]).clamp(min=0)

    union = area1 + area2 - intersection
    return intersection / union.clamp(min=1e-7)

## 3. Compare ground truth with simulated predictions

Create slightly shifted and poorly aligned predictions around the annotated apple. Their IoU values show how box alignment affects overlap.

In [ ]:
x1, y1, x2, y2 = ground_truth[0].tolist()
simulated_predictions = torch.tensor([
    [x1 + 5, y1 + 5, x2 - 5, y2 - 5],       # close prediction
    [x1 + 30, y1 + 20, x2 + 30, y2 + 20],   # shifted prediction
    [x2 + 20, y2 + 20, x2 + 100, y2 + 100]  # no overlap
], dtype=torch.float32)

ious = calculate_iou(simulated_predictions, ground_truth.expand_as(simulated_predictions))
for i, value in enumerate(ious, start=1):
    print(f"Prediction {i} IoU: {value.item():.4f}")

## 4. Visualize the boxes

The green rectangle is the annotated apple. Red rectangles are the simulated predictions.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))
ax.imshow(image)

gt = ground_truth[0].tolist()
ax.add_patch(patches.Rectangle((gt[0], gt[1]), gt[2]-gt[0], gt[3]-gt[1],
                               linewidth=2, edgecolor="lime", facecolor="none",
                               label="Ground truth"))

for i, box in enumerate(simulated_predictions.tolist()):
    ax.add_patch(patches.Rectangle((box[0], box[1]), box[2]-box[0], box[3]-box[1],
                                   linewidth=2, edgecolor="red", facecolor="none",
                                   label=f"Prediction {i+1}"))

ax.set_title("Apple Annotation vs Simulated Predictions")
ax.legend()
plt.axis("off")
plt.show()

### Day 13 · Key takeaway

IoU measures bounding-box overlap. A prediction that aligns closely with the annotated object has a higher IoU; a box with no overlap has an IoU of zero.